# Notebook ٢: Python اللي بتحتاجه في AI

مبني حوالين شات بوت شخصيات تاريخية (زي Historical-ChatBot بتاعك). كل خلية اتشغلت فعلًا.

## ١. JSON وقواميس: داتا الشات بوت

الشات بوت بتاعك شغال على ملفات JSON لشخصيات. هنعمل نسخة صغيرة منها.

In [ ]:
# مكتبة التعامل مع JSON
import json
# قاموس فيه الشخصيات
figures = {
# شخصية أولى
    "salah": {"name": "Mohamed Salah", "club": "Liverpool", "facts": ["Egyptian King", "Won the Golden Boot"]},
# شخصية تانية
    "zidane": {"name": "Zinedine Zidane", "club": "Real Madrid", "facts": ["Won the World Cup 1998"]},
# نقفل القاموس
}
# نفتح ملف للكتابة
with open("figures.json", "w") as f:
# نكتب القاموس كـ JSON منسق
    json.dump(figures, f, indent=2)
# نفتح الملف للقراءة
with open("figures.json") as f:
# نقرأه رجع قاموس
    loaded = json.load(f)
# نطبع أول معلومة عن صلاح
print(loaded["salah"]["facts"][0])


## ٢. Comprehensions

طريقة مختصرة لبناء قوايم وقواميس. بتتكرر جدًا في تجهيز الداتا.

In [ ]:
# قايمة بأسماء كل الشخصيات
names = [v["name"] for v in loaded.values()]
# نطبع الأسماء
print(names)
# قاموس من المفتاح للنادي
clubs = {k: v["club"] for k, v in loaded.items()}
# نطبع القاموس
print(clubs)
# المعلومات الأطول من 20 حرف
long_facts = [f for v in loaded.values() for f in v["facts"] if len(f) > 20]
# نطبعها
print(long_facts)


## ٣. دوال مع type hints و default

الدوال الواضحة هي أساس أي pipeline.

In [ ]:
# دالة تنضف نص، والتحويل لحروف صغيرة اختياري
def clean(text: str, lower: bool = True) -> str:
# نشيل المسافات من الأطراف
    text = text.strip()
# نرجع النص بالشكل المطلوب
    return text.lower() if lower else text

# نجرب مع الافتراضي
print(clean("  Who is SALAH?  "))
# نجرب من غير تصغير
print(clean("  Who is SALAH?  ", lower=False))


## ٤. Regex: استخراج من النص

مهم جدًا في NLP لتنضيف الرسائل واستخراج الكيانات.

In [ ]:
# مكتبة regex
import re
# رسالة مستخدم
msg = "Tell me about Salah's goals in 2018, 2022 and 2025"
# نستخرج أي سنة تبدأ بـ 20
years = re.findall(r"\b(20\d{2})\b", msg)
# نطبع السنين
print(years)
# نقطّع النص لكلمات
tokens = re.findall(r"[a-z']+", msg.lower())
# نطبع الكلمات
print(tokens)


## ٥. كلاس: شات بوت بسيط

نجمع الداتا والمنطق في كلاس واحد.

In [ ]:
# كلاس الشات بوت
class FigureBot:
# الدالة اللي بتشتغل أول ما نعمل كائن
    def __init__(self, data):
# نخزن الداتا جوه الكائن
        self.data = data

# بتدور على الشخصية المذكورة في الرسالة
    def find(self, text):
# نصغّر الرسالة
        text = text.lower()
# نلف على الشخصيات
        for key, info in self.data.items():
# لو اسمها موجود في الرسالة
            if key in text:
# نرجع بياناتها
                return info
# مفيش شخصية مطابقة
        return None

# بتولد الرد
    def reply(self, text):
# ندور على الشخصية
        info = self.find(text)
# لو ملقيناش
        if info is None:
# رد افتراضي
            return "I don't know this person."
# الرد بصيغة الشخصية
        return f"I am {info['name']} from {info['club']}. {info['facts'][0]}."

# ننشئ كائن البوت
bot = FigureBot(loaded)
# نجرب سؤال لصلاح
print(bot.reply("Hi Salah, who are you?"))
# نجرب شخصية مش موجودة
print(bot.reply("Tell me about Messi"))


## ٦. sorted و lambda و zip

ترتيب وربط قوايم من غير loops كتير.

In [ ]:
# درجات تشابه تجريبية
scores = {"salah": 0.91, "zidane": 0.78, "messi": 0.95}
# نرتب من الأعلى درجة
ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
# نطبع الترتيب
print(ranked)
# نرقّم النتايج من 1
for rank, (name, s) in enumerate(ranked, start=1):
# نطبع الترتيب والاسم والدرجة
    print(rank, name, s)
# ندمج قايمتين في قاموس
labels = dict(zip(["a", "b"], [0, 1]))
# نطبعه
print(labels)


## ٧. Generators و batching

في الداتا الكبيرة بنقسمها دفعات (batches) من غير ما نحمل كل حاجة في الذاكرة.

In [ ]:
# دالة generator بتطلع دفعات
def batches(items, size):
# نمشي بخطوة = حجم الدفعة
    for i in range(0, len(items), size):
# نطلع الدفعة الحالية
        yield items[i:i + size]

# بيانات تجريبية من 1 لـ 7
data = list(range(1, 8))
# نلف على الدفعات
for b in batches(data, 3):
# نطبع كل دفعة
    print(b)


## ٨. try/except و requests

التعامل مع API حقيقي لازم يتوقع الأخطاء.

In [ ]:
# مكتبة طلبات HTTP
import requests
# دالة بتجيب JSON من API
def get_json(url):
# نجرب الطلب
    try:
# نبعت GET
        r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
# نرمي خطأ لو الكود مش 200
        r.raise_for_status()
# نرجع الـ JSON
        return r.json()
# لو حصل أي خطأ شبكة
    except requests.RequestException as e:
# نرجع رسالة الخطأ
        return {"error": str(e)}

# نكلم FPL API
res = get_json("https://fantasy.premierleague.com/api/bootstrap-static/")
# نعد اللاعبين
print(len(res["elements"]), "players")
# رابط غلط عمدًا
bad = get_json("https://fantasy.premierleague.com/api/nonexistent-endpoint/")
# هل رجع خطأ؟
print("error" in bad)
